# Final drilling advisory model: near_5 + light penalty

Вход: `united_rock_energy_segment_quantile.csv`, полученный из `rock_energy_segment_log_pseudo_mse_only.ipynb`.

Notebook обучает simulator-ready advisory-модели:

1. `rotation_model_near5`: прогноз средней rotation на следующих 5 telemetry-точках;
2. `speed_model_near5`: прогноз средней speed на следующих 5 telemetry-точках;
3. light-penalty optimizer: перебор candidate `pressure_axis / pressure_rotation` и выбор реалистичной рекомендации.

Эта версия совместима с новой разметкой энергоёмкости, где `hardness_score_smooth` считается только по `log_pseudo_mse`.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred):
        return mean_squared_error(y_true, y_pred, squared=False)

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except Exception:
    HAS_LIGHTGBM = False

import joblib

RANDOM_STATE = 42
EPS = 1e-6
TARGET_HORIZON = 5

GRID_SIZE = 21
MAX_DELTA_FRAC = 0.08

FINAL_OPTIMIZER_MODE = "light_penalty"
CHANGE_PENALTY_WEIGHT = 0.010
BOUNDARY_PENALTY_WEIGHT = 0.020
BOUNDARY_START = 0.85

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Load prepared rock-energy dataset


In [2]:
CANDIDATE_PATHS = [
    "united_rock_energy_segment_quantile.csv",
    "../united_rock_energy_segment_quantile.csv",
    "notebooks/united_rock_energy_segment_quantile.csv",
    "../notebooks/united_rock_energy_segment_quantile.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if Path(p).exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Не найден united_rock_energy_segment_quantile.csv. "
        "Сначала запусти notebook rock_energy_segment_log_pseudo_mse_only.ipynb."
    )

df = pd.read_csv(DATA_PATH).drop(columns=["Unnamed: 0"], errors="ignore")

required_cols = [
    "processing_time",
    "well_id",
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "speed",
    "hardness_score_smooth",
    "rock_energy_type_final",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Не хватает колонок: {missing}")

df["processing_time"] = pd.to_datetime(df["processing_time"])
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)
df["rock_energy_type_final"] = df["rock_energy_type_final"].fillna("unknown").astype(str)

print("Loaded:", DATA_PATH)
print("Shape:", df.shape)

display(df[required_cols].head())
display(df[["pressure_axis", "pressure_rotation", "rotation", "speed", "hardness_score_smooth"]].describe(percentiles=[.01, .05, .5, .95, .99]))

Loaded: united_rock_energy_segment_quantile.csv
Shape: (415049, 89)


,processing_time,well_id,pressure_axis,pressure_rotation,rotation,speed,hardness_score_smooth,rock_energy_type_final
0,2025-08-24 10:09:50.980,19601,804,4551,74.256,0.002755,NaN,unknown
1,2025-08-24 10:10:00.260,19601,763,3782,73.812,0.003030,NaN,unknown
2,2025-08-24 10:10:05.199,19601,879,4407,73.512,0.006060,NaN,unknown
3,2025-08-24 10:10:14.610,19601,721,3705,73.962,0.002755,NaN,unknown
4,2025-08-24 10:10:34.197,19601,859,3883,74.256,0.001515,NaN,unknown


,pressure_axis,pressure_rotation,rotation,speed,hardness_score_smooth
count,415049.000000,415049.000000,415049.000000,415049.000000,3.826350e+05
mean,17473.667273,14134.146325,103.945315,0.013116,4.278465e-15
std,4682.982816,3243.523440,13.370361,0.006615,9.999975e-01
min,317.000000,784.000000,50.010000,0.001002,-6.033536e+00
1%,3713.000000,6271.000000,64.980000,0.002755,-2.697314e+00
5%,6645.000000,8246.000000,81.750000,0.005050,-1.737829e+00
50%,18861.000000,14637.000000,103.158000,0.012120,8.954485e-02
95%,22343.000000,18758.600000,138.474000,0.024240,1.660294e+00
99%,23626.000000,20881.000000,139.020000,0.030300,2.115928e+00
max,24872.000000,26318.000000,139.578000,0.038957,3.724039e+00


## 2. Feature engineering


In [3]:
def add_features(data):
    out = data.copy()

    out["total_pressure"] = out["pressure_axis"] + out["pressure_rotation"]
    out["pressure_balance"] = out["pressure_axis"] / (out["total_pressure"] + EPS)
    out["axis_over_rot_pressure"] = out["pressure_axis"] / (out["pressure_rotation"] + EPS)
    out["rot_pressure_over_axis"] = out["pressure_rotation"] / (out["pressure_axis"] + EPS)
    out["rotation_efficiency"] = out["rotation"] / (out["pressure_rotation"] + EPS)
    out["axis_x_rotation"] = out["pressure_axis"] * out["rotation"]
    out["rot_pressure_x_rotation"] = out["pressure_rotation"] * out["rotation"]
    out["energy_input_proxy"] = out["pressure_axis"] + out["pressure_rotation"] * out["rotation"]
    out["log_energy_input_proxy"] = np.log1p(out["energy_input_proxy"])

    out["dt"] = out.groupby("well_id")["processing_time"].diff().dt.total_seconds()
    out["dt"] = out["dt"].fillna(out["dt"].median())

    history_cols = [
        "pressure_axis",
        "pressure_rotation",
        "rotation",
        "speed",
        "hardness_score_smooth",
        "energy_input_proxy",
        "pressure_balance",
    ]

    for col in history_cols:
        for lag in [1, 3, 6, 12]:
            out[f"{col}_lag{lag}"] = out.groupby("well_id")[col].shift(lag)

        shifted = out.groupby("well_id")[col].shift(1)
        for w in [6, 12, 30]:
            min_p = max(2, w // 3)
            out[f"{col}_roll_mean_{w}"] = (
                shifted.groupby(out["well_id"])
                       .rolling(w, min_periods=min_p)
                       .mean()
                       .reset_index(level=0, drop=True)
            )
            out[f"{col}_roll_std_{w}"] = (
                shifted.groupby(out["well_id"])
                       .rolling(w, min_periods=min_p)
                       .std()
                       .reset_index(level=0, drop=True)
            )

    for col in ["pressure_axis", "pressure_rotation", "rotation", "speed", "hardness_score_smooth"]:
        prev = out.groupby("well_id")[col].shift(1)
        out[f"{col}_diff1"] = out[col] - prev
        out[f"{col}_rel_diff1"] = out[f"{col}_diff1"] / (prev.abs() + EPS)

    return out


df = add_features(df)
display(df.head())

,processing_time,depth_m,rotation,pressure_axis,pressure_rotation,well_id,speed,dt,total_pressure,pressure_balance,axis_over_rot_pressure,rot_pressure_over_axis,rotation_efficiency,axis_x_rotation,rot_pressure_x_rotation,energy_input_proxy,pseudo_mse,drilling_efficiency,log_energy_input_proxy,log_pseudo_mse,energy_input_proxy_roll_median_12,energy_input_proxy_roll_mean_12,energy_input_proxy_roll_std_12,energy_input_proxy_roll_median_30,energy_input_proxy_roll_mean_30,energy_input_proxy_roll_std_30,energy_input_proxy_roll_median_60,energy_input_proxy_roll_mean_60,energy_input_proxy_roll_std_60,pseudo_mse_roll_median_12,pseudo_mse_roll_mean_12,pseudo_mse_roll_std_12,pseudo_mse_roll_median_30,pseudo_mse_roll_mean_30,pseudo_mse_roll_std_30,pseudo_mse_roll_median_60,pseudo_mse_roll_mean_60,pseudo_mse_roll_std_60,drilling_efficiency_roll_median_12,drilling_efficiency_roll_mean_12,drilling_efficiency_roll_std_12,drilling_efficiency_roll_median_30,drilling_efficiency_roll_mean_30,drilling_efficiency_roll_std_30,drilling_efficiency_roll_median_60,drilling_efficiency_roll_mean_60,drilling_efficiency_roll_std_60,rotation_efficiency_roll_median_12,rotation_efficiency_roll_mean_12,rotation_efficiency_roll_std_12,rotation_efficiency_roll_median_30,rotation_efficiency_roll_mean_30,rotation_efficiency_roll_std_30,rotation_efficiency_roll_median_60,rotation_efficiency_roll_mean_60,rotation_efficiency_roll_std_60,speed_roll_median_12,speed_roll_mean_12,speed_roll_std_12,speed_roll_median_30,speed_roll_mean_30,speed_roll_std_30,speed_roll_median_60,speed_roll_mean_60,speed_roll_std_60,rotation_roll_median_12,rotation_roll_mean_12,rotation_roll_std_12,rotation_roll_median_30,rotation_roll_mean_30,rotation_roll_std_30,rotation_roll_median_60,rotation_roll_mean_60,rotation_roll_std_60,pressure_balance_roll_median_12,pressure_balance_roll_mean_12,pressure_balance_roll_std_12,pressure_balance_roll_median_30,pressure_balance_roll_mean_30,pressure_balance_roll_std_30,pressure_balance_roll_median_60,pressure_balance_roll_mean_60,pressure_balance_roll_std_60,hardness_score,hardness_score_smooth,segment_id,hardness_segment,energy_type_segment_quantile,rock_energy_type_final,pressure_axis_lag1,pressure_axis_lag3,pressure_axis_lag6,pressure_axis_lag12,pressure_axis_roll_mean_6,pressure_axis_roll_std_6,pressure_axis_roll_mean_12,pressure_axis_roll_std_12,pressure_axis_roll_mean_30,pressure_axis_roll_std_30,pressure_rotation_lag1,pressure_rotation_lag3,pressure_rotation_lag6,pressure_rotation_lag12,pressure_rotation_roll_mean_6,pressure_rotation_roll_std_6,pressure_rotation_roll_mean_12,pressure_rotation_roll_std_12,pressure_rotation_roll_mean_30,pressure_rotation_roll_std_30,rotation_lag1,rotation_lag3,rotation_lag6,rotation_lag12,rotation_roll_mean_6,rotation_roll_std_6,speed_lag1,speed_lag3,speed_lag6,speed_lag12,speed_roll_mean_6,speed_roll_std_6,hardness_score_smooth_lag1,hardness_score_smooth_lag3,hardness_score_smooth_lag6,hardness_score_smooth_lag12,hardness_score_smooth_roll_mean_6,hardness_score_smooth_roll_std_6,hardness_score_smooth_roll_mean_12,hardness_score_smooth_roll_std_12,hardness_score_smooth_roll_mean_30,hardness_score_smooth_roll_std_30,energy_input_proxy_lag1,energy_input_proxy_lag3,energy_input_proxy_lag6,energy_input_proxy_lag12,energy_input_proxy_roll_mean_6,energy_input_proxy_roll_std_6,pressure_balance_lag1,pressure_balance_lag3,pressure_balance_lag6,pressure_balance_lag12,pressure_balance_roll_mean_6,pressure_balance_roll_std_6,pressure_axis_diff1,pressure_axis_rel_diff1,pressure_rotation_diff1,pressure_rotation_rel_diff1,rotation_diff1,rotation_rel_diff1,speed_diff1,speed_rel_diff1,hardness_score_smooth_diff1,hardness_score_smooth_rel_diff1
0,2025-08-24 10:09:50.980,0.0606,74.256,804,4551,19601,0.002755,5.135,5355,0.150140,0.176664,5.660448,0.016316,59701.824,337939.056,338743.056,1.229314e+08,8.131666e-09,12.733000,18.627137,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

## 3. near_5 targets


In [4]:
def future_mean_by_group(data, value_col, horizon):
    return (
        data.groupby("well_id")[value_col]
            .transform(
                lambda s: (
                    s.shift(-1)
                     .rolling(horizon, min_periods=max(2, horizon // 3))
                     .mean()
                     .shift(-(horizon - 1))
                )
            )
    )

df["target_rotation_near5"] = future_mean_by_group(df, "rotation", TARGET_HORIZON)
df["target_speed_near5"] = future_mean_by_group(df, "speed", TARGET_HORIZON)

target_rotation = "target_rotation_near5"
target_speed = "target_speed_near5"

display(df[[target_rotation, target_speed, "rotation", "speed"]].describe(percentiles=[.01, .05, .5, .95, .99]))

,target_rotation_near5,target_speed_near5,rotation,speed
count,408225.000000,408225.000000,415049.000000,415049.000000
mean,104.056793,0.013110,103.945315,0.013116
std,10.170965,0.005407,13.370361,0.006615
min,50.868000,0.001409,50.010000,0.001002
1%,68.893104,0.003814,64.980000,0.002755
5%,85.029600,0.005656,81.750000,0.005050
50%,103.308000,0.012524,103.158000,0.012120
95%,123.786000,0.023230,138.474000,0.024240
99%,128.200560,0.028280,139.020000,0.030300
max,139.317600,0.036360,139.578000,0.038957


## 4. Feature lists


In [5]:
base_numeric_features = [
    "pressure_axis", "pressure_rotation", "total_pressure", "pressure_balance",
    "axis_over_rot_pressure", "rot_pressure_over_axis",
    "rotation", "speed", "hardness_score_smooth", "dt",
    "rotation_efficiency", "axis_x_rotation", "rot_pressure_x_rotation",
    "energy_input_proxy", "log_energy_input_proxy",
    "pressure_axis_lag1", "pressure_axis_lag3", "pressure_axis_lag6",
    "pressure_rotation_lag1", "pressure_rotation_lag3", "pressure_rotation_lag6",
    "rotation_lag1", "rotation_lag3", "rotation_lag6",
    "speed_lag1", "speed_lag3", "speed_lag6",
    "hardness_score_smooth_lag1", "hardness_score_smooth_lag3", "hardness_score_smooth_lag6",
    "pressure_axis_roll_mean_12", "pressure_axis_roll_std_12",
    "pressure_rotation_roll_mean_12", "pressure_rotation_roll_std_12",
    "rotation_roll_mean_12", "rotation_roll_std_12",
    "speed_roll_mean_12", "speed_roll_std_12",
    "hardness_score_smooth_roll_mean_12", "hardness_score_smooth_roll_std_12",
    "pressure_axis_diff1", "pressure_rotation_diff1", "rotation_diff1",
    "speed_diff1", "hardness_score_smooth_diff1",
]

categorical_features = ["rock_energy_type_final"]

speed_extra_features = ["candidate_target_rotation"]
speed_numeric_features = base_numeric_features + speed_extra_features

all_required = base_numeric_features + categorical_features + [target_rotation, target_speed]
missing = [c for c in all_required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

model_df = df.dropna(subset=all_required + ["well_id", "processing_time"]).copy()

print("Model df:", model_df.shape)
print("Numeric features:", len(base_numeric_features))
print("Categorical features:", categorical_features)

Model df: (365575, 155)
Numeric features: 45
Categorical features: ['rock_energy_type_final']


## 5. Split by unseen wells


In [6]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(model_df, groups=model_df["well_id"]))

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Train:", train_df.shape, "wells:", train_df["well_id"].nunique())
print("Test:", test_df.shape, "wells:", test_df["well_id"].nunique())
print("Test wells:", list(test_df["well_id"].unique())[:10])

Train: (271790, 155) wells: 1279
Test: (93785, 155) wells: 427
Test wells: [np.int64(19677), np.int64(19720), np.int64(19775), np.int64(19780), np.int64(19789), np.int64(19905), np.int64(19909), np.int64(19935), np.int64(19958), np.int64(19977)]


## 6. Train models


In [7]:
def make_regressor():
    if HAS_LIGHTGBM:
        return lgb.LGBMRegressor(
            n_estimators=650,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=RANDOM_STATE,
            objective="regression",
            verbosity=-1,
        )

    return HistGradientBoostingRegressor(
        max_iter=500,
        learning_rate=0.04,
        max_leaf_nodes=63,
        l2_regularization=0.01,
        random_state=RANDOM_STATE,
    )


def make_preprocessor(numeric_features, categorical_features):
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            ("num", "passthrough", numeric_features),
        ],
        remainder="drop",
    )


def regression_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(root_mean_squared_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }


rotation_model = Pipeline(steps=[
    ("preprocess", make_preprocessor(base_numeric_features, categorical_features)),
    ("model", make_regressor()),
])

rotation_model.fit(
    train_df[base_numeric_features + categorical_features],
    train_df[target_rotation],
)

test_df["pred_target_rotation"] = rotation_model.predict(
    test_df[base_numeric_features + categorical_features]
)

rotation_metrics = regression_metrics(
    test_df[target_rotation],
    test_df["pred_target_rotation"],
)

print("Rotation model near_5:")
print(rotation_metrics)


train_speed_df = train_df.copy()
train_speed_df["candidate_target_rotation"] = train_speed_df[target_rotation]

test_speed_oracle_df = test_df.copy()
test_speed_oracle_df["candidate_target_rotation"] = test_speed_oracle_df[target_rotation]

test_speed_chained_df = test_df.copy()
test_speed_chained_df["candidate_target_rotation"] = test_speed_chained_df["pred_target_rotation"]

speed_model = Pipeline(steps=[
    ("preprocess", make_preprocessor(speed_numeric_features, categorical_features)),
    ("model", make_regressor()),
])

speed_model.fit(
    train_speed_df[speed_numeric_features + categorical_features],
    train_speed_df[target_speed],
)

test_df["pred_target_speed_oracle_rotation"] = speed_model.predict(
    test_speed_oracle_df[speed_numeric_features + categorical_features]
)

test_df["pred_target_speed_chained"] = speed_model.predict(
    test_speed_chained_df[speed_numeric_features + categorical_features]
)

speed_metrics_oracle = regression_metrics(
    test_df[target_speed],
    test_df["pred_target_speed_oracle_rotation"],
)

speed_metrics_chained = regression_metrics(
    test_df[target_speed],
    test_df["pred_target_speed_chained"],
)

print("Speed model near_5 with actual target rotation:")
print(speed_metrics_oracle)

print("Speed model near_5 chained:")
print(speed_metrics_chained)

Rotation model near_5:
{'MAE': 1.7424399885057065, 'RMSE': 3.5934602072791875, 'R2': 0.8172378197318995}
Speed model near_5 with actual target rotation:
{'MAE': 0.001938000000159906, 'RMSE': 0.0026247322173224568, 'R2': 0.7543863873089091}
Speed model near_5 chained:
{'MAE': 0.0020307206672383338, 'RMSE': 0.0027360723168897466, 'R2': 0.7331067605298305}


## 7. Baselines and surface ranges


In [8]:
test_df["baseline_current_speed"] = test_df["speed"]
test_df["baseline_speed_roll_mean_12"] = test_df["speed_roll_mean_12"].fillna(test_df["speed"])

baseline_compare = pd.DataFrame([
    {"model": "current_speed", **regression_metrics(test_df[target_speed], test_df["baseline_current_speed"])},
    {"model": "speed_roll_mean_12", **regression_metrics(test_df[target_speed], test_df["baseline_speed_roll_mean_12"])},
    {"model": "rotation_to_speed_chained_near5", **speed_metrics_chained},
    {"model": "speed_oracle_rotation_near5", **speed_metrics_oracle},
]).sort_values("MAE")

display(baseline_compare)

surface_ranges = {}

for et, part in train_df.groupby("rock_energy_type_final"):
    if len(part) < 100:
        continue

    surface_ranges[et] = {
        "pressure_axis_q05": float(part["pressure_axis"].quantile(0.05)),
        "pressure_axis_q95": float(part["pressure_axis"].quantile(0.95)),
        "pressure_rotation_q05": float(part["pressure_rotation"].quantile(0.05)),
        "pressure_rotation_q95": float(part["pressure_rotation"].quantile(0.95)),
        "rotation_median": float(part["rotation"].median()),
        "speed_median": float(part["speed"].median()),
        "hardness_median": float(part["hardness_score_smooth"].median()),
        "rows": int(len(part)),
    }

surface_ranges_df = pd.DataFrame(surface_ranges).T
display(surface_ranges_df)

,model,MAE,RMSE,R2
3,speed_oracle_rotation_near5,0.001938,0.002625,0.754386
2,rotation_to_speed_chained_near5,0.002031,0.002736,0.733107
1,speed_roll_mean_12,0.002483,0.003403,0.587099
0,current_speed,0.003512,0.004528,0.268915


,pressure_axis_q05,pressure_axis_q95,pressure_rotation_q05,pressure_rotation_q95,rotation_median,speed_median,hardness_median,rows
hard_high_energy,13522.0,22891.0,9896.0,18836.00,103.458,0.00606,1.111576,67303.0
medium_high_energy,14058.0,22685.0,10469.0,18881.45,102.864,0.01212,0.311459,69412.0
medium_low_energy,12547.0,22352.8,9755.2,19057.00,102.966,0.01212,-0.213769,68165.0
soft_low_energy,7527.0,21376.0,8315.0,18579.55,103.410,0.01818,-1.071588,66910.0


## 8. Final light-penalty optimizer


In [9]:
def recompute_candidate_features(grid):
    grid = grid.copy()
    grid["total_pressure"] = grid["pressure_axis"] + grid["pressure_rotation"]
    grid["pressure_balance"] = grid["pressure_axis"] / (grid["total_pressure"] + EPS)
    grid["axis_over_rot_pressure"] = grid["pressure_axis"] / (grid["pressure_rotation"] + EPS)
    grid["rot_pressure_over_axis"] = grid["pressure_rotation"] / (grid["pressure_axis"] + EPS)
    grid["rotation_efficiency"] = grid["rotation"] / (grid["pressure_rotation"] + EPS)
    grid["axis_x_rotation"] = grid["pressure_axis"] * grid["rotation"]
    grid["rot_pressure_x_rotation"] = grid["pressure_rotation"] * grid["rotation"]
    grid["energy_input_proxy"] = grid["pressure_axis"] + grid["pressure_rotation"] * grid["rotation"]
    grid["log_energy_input_proxy"] = np.log1p(grid["energy_input_proxy"])
    return grid


def build_candidate_grid(row, grid_size=GRID_SIZE, max_delta_frac=MAX_DELTA_FRAC):
    et = row["rock_energy_type_final"]

    if et in surface_ranges:
        r = surface_ranges[et]
        p_ax_low = r["pressure_axis_q05"]
        p_ax_high = r["pressure_axis_q95"]
        p_rot_low = r["pressure_rotation_q05"]
        p_rot_high = r["pressure_rotation_q95"]
    else:
        p_ax_low = train_df["pressure_axis"].quantile(0.05)
        p_ax_high = train_df["pressure_axis"].quantile(0.95)
        p_rot_low = train_df["pressure_rotation"].quantile(0.05)
        p_rot_high = train_df["pressure_rotation"].quantile(0.95)

    cur_ax = row["pressure_axis"]
    cur_rotp = row["pressure_rotation"]

    local_ax_low = cur_ax * (1.0 - max_delta_frac)
    local_ax_high = cur_ax * (1.0 + max_delta_frac)
    local_rotp_low = cur_rotp * (1.0 - max_delta_frac)
    local_rotp_high = cur_rotp * (1.0 + max_delta_frac)

    p_ax_min = max(p_ax_low, local_ax_low)
    p_ax_max = min(p_ax_high, local_ax_high)
    p_rot_min = max(p_rot_low, local_rotp_low)
    p_rot_max = min(p_rot_high, local_rotp_high)

    if p_ax_min >= p_ax_max:
        p_ax_min, p_ax_max = local_ax_low, local_ax_high

    if p_rot_min >= p_rot_max:
        p_rot_min, p_rot_max = local_rotp_low, local_rotp_high

    p_ax_grid = np.linspace(p_ax_min, p_ax_max, grid_size)
    p_rot_grid = np.linspace(p_rot_min, p_rot_max, grid_size)
    PA, PR = np.meshgrid(p_ax_grid, p_rot_grid)

    grid = pd.DataFrame({"pressure_axis": PA.ravel(), "pressure_rotation": PR.ravel()})

    recomputed = {
        "pressure_axis", "pressure_rotation", "total_pressure", "pressure_balance",
        "axis_over_rot_pressure", "rot_pressure_over_axis", "rotation_efficiency",
        "axis_x_rotation", "rot_pressure_x_rotation", "energy_input_proxy",
        "log_energy_input_proxy",
    }

    for col in base_numeric_features:
        if col not in recomputed:
            grid[col] = row[col]

    for col in categorical_features:
        grid[col] = row[col]

    grid = recompute_candidate_features(grid)

    grid["delta_pressure_axis_frac"] = grid["pressure_axis"] / (row["pressure_axis"] + EPS) - 1.0
    grid["delta_pressure_rotation_frac"] = grid["pressure_rotation"] / (row["pressure_rotation"] + EPS) - 1.0

    return grid, PA, PR


def predict_current_target_speed(row):
    current_grid = pd.DataFrame([row[base_numeric_features + categorical_features].to_dict()])
    current_grid["candidate_target_rotation"] = rotation_model.predict(
        current_grid[base_numeric_features + categorical_features]
    )
    return float(speed_model.predict(current_grid[speed_numeric_features + categorical_features])[0])


def add_light_penalized_score(grid, current_pred_speed):
    grid = grid.copy()

    speed_scale = max(abs(current_pred_speed), EPS)

    axis_delta_norm = np.abs(grid["delta_pressure_axis_frac"]) / MAX_DELTA_FRAC
    rot_delta_norm = np.abs(grid["delta_pressure_rotation_frac"]) / MAX_DELTA_FRAC

    change_penalty = (
        CHANGE_PENALTY_WEIGHT
        * speed_scale
        * (axis_delta_norm**2 + rot_delta_norm**2)
        / 2.0
    )

    axis_edge = np.clip((axis_delta_norm - BOUNDARY_START) / (1.0 - BOUNDARY_START + EPS), 0, 1)
    rot_edge = np.clip((rot_delta_norm - BOUNDARY_START) / (1.0 - BOUNDARY_START + EPS), 0, 1)

    boundary_penalty = (
        BOUNDARY_PENALTY_WEIGHT
        * speed_scale
        * (axis_edge**2 + rot_edge**2)
        / 2.0
    )

    grid["change_penalty"] = change_penalty
    grid["boundary_penalty"] = boundary_penalty
    grid["optimizer_score"] = grid["pred_target_speed"] - change_penalty - boundary_penalty

    return grid


def recommend_for_row(row, grid_size=GRID_SIZE):
    grid, PA, PR = build_candidate_grid(row, grid_size=grid_size)

    grid["candidate_target_rotation"] = rotation_model.predict(
        grid[base_numeric_features + categorical_features]
    )

    grid["pred_target_speed"] = speed_model.predict(
        grid[speed_numeric_features + categorical_features]
    )

    current_pred_speed = predict_current_target_speed(row)

    grid = add_light_penalized_score(grid, current_pred_speed=current_pred_speed)

    best_idx = int(grid["optimizer_score"].values.argmax())
    best = grid.iloc[best_idx].copy()

    return {
        "recommended_pressure_axis": float(best["pressure_axis"]),
        "recommended_pressure_rotation": float(best["pressure_rotation"]),
        "predicted_target_rotation": float(best["candidate_target_rotation"]),
        "predicted_target_speed": float(best["pred_target_speed"]),
        "optimizer_score": float(best["optimizer_score"]),
        "change_penalty": float(best["change_penalty"]),
        "boundary_penalty": float(best["boundary_penalty"]),
        "current_predicted_target_speed": current_pred_speed,
        "predicted_uplift_pct": 100.0 * (float(best["pred_target_speed"]) / (current_pred_speed + EPS) - 1.0),
        "score_uplift_pct": 100.0 * (float(best["optimizer_score"]) / (current_pred_speed + EPS) - 1.0),
        "delta_pressure_axis_pct": 100.0 * (float(best["pressure_axis"]) / (row["pressure_axis"] + EPS) - 1.0),
        "delta_pressure_rotation_pct": 100.0 * (float(best["pressure_rotation"]) / (row["pressure_rotation"] + EPS) - 1.0),
        "grid": grid,
        "PA": PA,
        "PR": PR,
        "Z": grid["pred_target_speed"].values.reshape(PA.shape),
        "Z_score": grid["optimizer_score"].values.reshape(PA.shape),
    }

## 9. Offline replay evaluation


In [10]:
EVAL_N = min(2500, len(test_df))
eval_points = test_df.sample(EVAL_N, random_state=RANDOM_STATE).copy()

recommendations = []

for _, row in eval_points.iterrows():
    rec = recommend_for_row(row, grid_size=GRID_SIZE)

    recommendations.append({
        "well_id": row["well_id"],
        "processing_time": row["processing_time"],
        "rock_energy_type_final": row["rock_energy_type_final"],
        "operator_pressure_axis": row["pressure_axis"],
        "operator_pressure_rotation": row["pressure_rotation"],
        "recommended_pressure_axis": rec["recommended_pressure_axis"],
        "recommended_pressure_rotation": rec["recommended_pressure_rotation"],
        "current_speed": row["speed"],
        "target_speed_actual": row[target_speed],
        "current_predicted_target_speed": rec["current_predicted_target_speed"],
        "recommended_predicted_target_speed": rec["predicted_target_speed"],
        "optimizer_score": rec["optimizer_score"],
        "change_penalty": rec["change_penalty"],
        "boundary_penalty": rec["boundary_penalty"],
        "predicted_uplift_pct": rec["predicted_uplift_pct"],
        "score_uplift_pct": rec["score_uplift_pct"],
        "delta_pressure_axis_pct": rec["delta_pressure_axis_pct"],
        "delta_pressure_rotation_pct": rec["delta_pressure_rotation_pct"],
    })

rec_df = pd.DataFrame(recommendations)

boundary_tol = 0.95 * MAX_DELTA_FRAC * 100.0
rec_df["axis_near_boundary"] = rec_df["delta_pressure_axis_pct"].abs() >= boundary_tol
rec_df["rot_near_boundary"] = rec_df["delta_pressure_rotation_pct"].abs() >= boundary_tol
rec_df["any_boundary"] = rec_df["axis_near_boundary"] | rec_df["rot_near_boundary"]

optimizer_summary = pd.DataFrame([{
    "optimizer_mode": FINAL_OPTIMIZER_MODE,
    "rows": len(rec_df),
    "median_uplift_pct": rec_df["predicted_uplift_pct"].median(),
    "mean_uplift_pct": rec_df["predicted_uplift_pct"].mean(),
    "p05_uplift_pct": rec_df["predicted_uplift_pct"].quantile(0.05),
    "p95_uplift_pct": rec_df["predicted_uplift_pct"].quantile(0.95),
    "median_score_uplift_pct": rec_df["score_uplift_pct"].median(),
    "median_delta_axis_pct": rec_df["delta_pressure_axis_pct"].median(),
    "median_delta_rot_pct": rec_df["delta_pressure_rotation_pct"].median(),
    "median_abs_delta_axis_pct": rec_df["delta_pressure_axis_pct"].abs().median(),
    "median_abs_delta_rot_pct": rec_df["delta_pressure_rotation_pct"].abs().median(),
    "axis_boundary_ratio": rec_df["axis_near_boundary"].mean(),
    "rot_boundary_ratio": rec_df["rot_near_boundary"].mean(),
    "any_boundary_ratio": rec_df["any_boundary"].mean(),
}])

display(optimizer_summary)

by_energy = (
    rec_df
    .groupby("rock_energy_type_final")
    .agg(
        rows=("predicted_uplift_pct", "size"),
        median_uplift_pct=("predicted_uplift_pct", "median"),
        mean_uplift_pct=("predicted_uplift_pct", "mean"),
        p05_uplift_pct=("predicted_uplift_pct", lambda s: s.quantile(0.05)),
        p95_uplift_pct=("predicted_uplift_pct", lambda s: s.quantile(0.95)),
        median_delta_axis_pct=("delta_pressure_axis_pct", "median"),
        median_delta_rot_pct=("delta_pressure_rotation_pct", "median"),
        median_abs_delta_axis_pct=("delta_pressure_axis_pct", lambda s: s.abs().median()),
        median_abs_delta_rot_pct=("delta_pressure_rotation_pct", lambda s: s.abs().median()),
        boundary_ratio=("any_boundary", "mean"),
    )
    .reset_index()
)

display(by_energy)

,optimizer_mode,rows,median_uplift_pct,mean_uplift_pct,p05_uplift_pct,p95_uplift_pct,median_score_uplift_pct,median_delta_axis_pct,median_delta_rot_pct,median_abs_delta_axis_pct,median_abs_delta_rot_pct,axis_boundary_ratio,rot_boundary_ratio,any_boundary_ratio
0,light_penalty,2500,2.254833,2.894645,0.160361,7.830637,1.805866,-5.588940e-09,5.6,3.2,5.6,0.012,0.0332,0.0432


,rock_energy_type_final,rows,median_uplift_pct,mean_uplift_pct,p05_uplift_pct,p95_uplift_pct,median_delta_axis_pct,median_delta_rot_pct,median_abs_delta_axis_pct,median_abs_delta_rot_pct,boundary_ratio
0,hard_high_energy,655,4.235326,4.793787,0.577997,10.685040,-5.619871e-09,6.4,3.855583,6.400000,0.083969
1,medium_high_energy,584,2.891864,3.170359,0.353867,7.115728,-8.000000e-01,6.4,3.200000,6.400000,0.054795
2,medium_low_energy,661,1.660941,2.089285,0.249396,5.298113,-5.611045e-09,4.8,3.200000,5.600000,0.015129
3,soft_low_energy,600,0.914473,1.440290,-0.000977,4.602192,8.000000e-01,0.8,2.979039,2.562533,0.018333


## 10. Example and save simulator-ready artifacts


In [11]:
example_row = test_df.sample(1, random_state=RANDOM_STATE + 10).iloc[0]
rec = recommend_for_row(example_row, grid_size=31)

example_summary = pd.DataFrame([
    {
        "variant": "operator/current",
        "pressure_axis": example_row["pressure_axis"],
        "pressure_rotation": example_row["pressure_rotation"],
        "current_speed": example_row["speed"],
        "target_speed_actual": example_row[target_speed],
        "predicted_target_speed": rec["current_predicted_target_speed"],
        "predicted_uplift_pct": 0.0,
        "delta_axis_pct": 0.0,
        "delta_rot_pct": 0.0,
    },
    {
        "variant": FINAL_OPTIMIZER_MODE,
        "pressure_axis": rec["recommended_pressure_axis"],
        "pressure_rotation": rec["recommended_pressure_rotation"],
        "current_speed": example_row["speed"],
        "target_speed_actual": example_row[target_speed],
        "predicted_target_speed": rec["predicted_target_speed"],
        "optimizer_score": rec["optimizer_score"],
        "predicted_uplift_pct": rec["predicted_uplift_pct"],
        "score_uplift_pct": rec["score_uplift_pct"],
        "delta_axis_pct": rec["delta_pressure_axis_pct"],
        "delta_rot_pct": rec["delta_pressure_rotation_pct"],
    },
])

print("Example well:", example_row["well_id"])
print("Example time:", example_row["processing_time"])
print("Energy type:", example_row["rock_energy_type_final"])
display(example_summary)

ARTIFACT_DIR = Path("drilling_advisory_light_penalty_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

joblib.dump(rotation_model, ARTIFACT_DIR / "rotation_model_near5.joblib")
joblib.dump(speed_model, ARTIFACT_DIR / "speed_model_near5.joblib")

with open(ARTIFACT_DIR / "surface_ranges_by_energy_type.json", "w", encoding="utf-8") as f:
    json.dump(surface_ranges, f, ensure_ascii=False, indent=2)

feature_config = {
    "data_path": DATA_PATH,
    "rock_energy_segmentation_expected_method": "energy_type_segment_quantile_log_pseudo_mse_only",
    "target_horizon": TARGET_HORIZON,
    "target_rotation": target_rotation,
    "target_speed": target_speed,
    "base_numeric_features": base_numeric_features,
    "categorical_features": categorical_features,
    "speed_numeric_features": speed_numeric_features,
    "speed_extra_features": speed_extra_features,
    "energy_type_column": "rock_energy_type_final",
    "required_live_columns": [
        "processing_time", "well_id", "pressure_axis", "pressure_rotation",
        "rotation", "speed", "hardness_score_smooth", "rock_energy_type_final",
    ],
}

with open(ARTIFACT_DIR / "feature_config.json", "w", encoding="utf-8") as f:
    json.dump(feature_config, f, ensure_ascii=False, indent=2)

optimizer_config = {
    "optimizer_mode": FINAL_OPTIMIZER_MODE,
    "grid_size_default": GRID_SIZE,
    "max_delta_frac_default": MAX_DELTA_FRAC,
    "change_penalty_weight": CHANGE_PENALTY_WEIGHT,
    "boundary_penalty_weight": BOUNDARY_PENALTY_WEIGHT,
    "boundary_start": BOUNDARY_START,
    "score_formula": "pred_target_speed - change_penalty - boundary_penalty",
    "use_local_reachable_bounds": True,
    "use_energy_type_quantile_bounds": True,
}

with open(ARTIFACT_DIR / "optimizer_config.json", "w", encoding="utf-8") as f:
    json.dump(optimizer_config, f, ensure_ascii=False, indent=2)

training_report = {
    "rock_energy_segmentation_expected_method": "energy_type_segment_quantile_log_pseudo_mse_only",
    "rotation_model_near5": rotation_metrics,
    "speed_model_oracle_rotation_near5": speed_metrics_oracle,
    "speed_model_chained_near5": speed_metrics_chained,
    "baseline_compare": baseline_compare.to_dict(orient="records"),
    "optimizer_summary": optimizer_summary.to_dict(orient="records"),
    "uplift_by_energy_type": by_energy.to_dict(orient="records"),
    "final_optimizer_mode": FINAL_OPTIMIZER_MODE,
}

with open(ARTIFACT_DIR / "training_report.json", "w", encoding="utf-8") as f:
    json.dump(training_report, f, ensure_ascii=False, indent=2)

rec_df.to_csv(ARTIFACT_DIR / "offline_recommendations_light_penalty.csv", index=False)
optimizer_summary.to_csv(ARTIFACT_DIR / "optimizer_summary.csv", index=False)
by_energy.to_csv(ARTIFACT_DIR / "uplift_by_energy_type.csv", index=False)

print("Saved artifacts to:", ARTIFACT_DIR.resolve())
for p in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", p.name)

Example well: 26435
Example time: 2025-10-15 00:45:28.008000
Energy type: hard_high_energy


,variant,pressure_axis,pressure_rotation,current_speed,target_speed_actual,predicted_target_speed,predicted_uplift_pct,delta_axis_pct,delta_rot_pct,optimizer_score,score_uplift_pct
0,operator/current,19981.000000,15348.000,0.01818,0.01313,0.014412,0.000000,0.000000,0.000000,NaN,NaN
1,light_penalty,18808.781333,16248.416,0.01818,0.01313,0.014696,1.960851,-5.866667,5.866667,0.014618,1.423111


Saved artifacts to: C:\Users\qa1ro\OneDrive\Рабочий стол\diploma\OptimalDrilling\notebooks\drilling_advisory_light_penalty_artifacts
 - feature_config.json
 - offline_recommendations_light_penalty.csv
 - optimizer_config.json
 - optimizer_summary.csv
 - rotation_model_near5.joblib
 - speed_model_near5.joblib
 - surface_ranges_by_energy_type.json
 - training_report.json
 - uplift_by_energy_type.csv


## What simulator should load

Для подключения к симулятору нужны:

```text
drilling_advisory_light_penalty_artifacts/
    rotation_model_near5.joblib
    speed_model_near5.joblib
    feature_config.json
    optimizer_config.json
    surface_ranges_by_energy_type.json
```

В online/replay режиме симулятор должен хранить rolling buffer telemetry, создавать те же признаки, строить candidate grid `p_ax/p_rot`, считать `target_rotation_near5`, `target_speed_near5`, `optimizer_score`, выбирать recommended point и рисовать current point + future-speed surface + recommended point.
